# 🎯 Placemux Recommendation Engine — ML Pipeline

**End-to-end ML pipeline for the Placemux placement recommendation system.**

This notebook walks through every stage of the ML lifecycle:

| Stage | Description |
|-------|-------------|
| **1. Data Generation** | Synthetic dataset with realistic edge cases |
| **2. Exploratory Data Analysis** | Understanding distributions, class balance, and data quality |
| **3. Feature Engineering** | Skill overlap, proficiency gap, experience fit, college priors |
| **4. Data Splitting** | Group-aware train/val/test split (no student leakage) |
| **5. Baseline Model** | Rule-based ranker using skill overlap only |
| **6. Model Training** | GradientBoosting with hyperparameter sweep |
| **7. Evaluation** | Precision, Recall, FPR, AUC-ROC — global and per-segment |
| **8. Explainability** | Feature importance + plain-English explanations |
| **9. Model Persistence** | Save the best model artifact for serving |

---
## 0. Setup & Imports

In [ ]:
import os
import sys
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay, classification_report
)

warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3
})
sns.set_style('whitegrid')
palette = sns.color_palette('viridis', 10)

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print("✅ All imports loaded successfully.")

---
## 1. Data Generation

We generate a realistic-scale synthetic dataset simulating a multi-college placement platform.

**Key design decisions:**
- 5,000 students across 20 colleges
- 500 jobs across 80 companies
- 150 skills in the taxonomy
- **Edge cases injected**: 50 cold-start students (zero skills), 10 zero-requirement jobs, ~4% missing proficiency scores, ~100 students with duplicate skill entries

In [ ]:
# Configuration
NUM_COLLEGES = 20
NUM_STUDENTS = 5000
NUM_COMPANIES = 80
NUM_JOBS = 500
NUM_SKILLS = 150

np.random.seed(42)
random.seed(42)

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# Taxonomy
skills_taxonomy = [f"skill_{i}" for i in range(NUM_SKILLS)]
colleges = [f"college_{i}" for i in range(NUM_COLLEGES)]
companies = [f"company_{i}" for i in range(NUM_COMPANIES)]

print(f"📊 Scale: {NUM_STUDENTS} students | {NUM_JOBS} jobs | {NUM_SKILLS} skills | {NUM_COLLEGES} colleges | {NUM_COMPANIES} companies")

In [ ]:
# --- 1a. Generate Students ---
student_records = []
for i in range(NUM_STUDENTS):
    student_records.append({
        "student_id": f"student_{i}",
        "college_id": random.choice(colleges),
        "years_of_experience": max(0, int(np.random.normal(2, 1.5))),
        "degree": random.choice(["Bachelors", "Masters", "PhD", "None"])
    })
students_df = pd.DataFrame(student_records)
students_df.to_csv(f"{DATA_DIR}/students.csv", index=False)

# --- 1b. Generate Student Skills (with edge cases) ---
student_skills_records = []

cold_start_students = set(random.sample(range(NUM_STUDENTS), 50))   # 50 students with zero skills
duplicate_skill_students = set(random.sample(range(NUM_STUDENTS), 100))  # 100 students have duplicates

for i in range(NUM_STUDENTS):
    if i in cold_start_students:
        continue
    num_skills = random.randint(10, 30)
    s_skills = random.sample(skills_taxonomy, num_skills)
    for skill in s_skills:
        prof = random.randint(10, 100)
        if random.random() < 0.04:  # ~4% missing proficiency
            prof = np.nan
        student_skills_records.append({"student_id": f"student_{i}", "skill_id": skill, "proficiency": prof})
        # Inject conflicting duplicates
        if i in duplicate_skill_students and random.random() < 0.1:
            student_skills_records.append({
                "student_id": f"student_{i}", "skill_id": skill,
                "proficiency": prof - 5 if pd.notnull(prof) else np.nan
            })

student_skills_df = pd.DataFrame(student_skills_records)
student_skills_df.to_csv(f"{DATA_DIR}/student_skills.csv", index=False)

print(f"✅ Students: {len(students_df):,}")
print(f"✅ Student Skills entries: {len(student_skills_df):,}")
print(f"   ├─ Missing proficiency: {student_skills_df['proficiency'].isna().sum()}")
print(f"   ├─ Cold-start students (zero skills): {len(cold_start_students)}")
print(f"   └─ Students with duplicate skills: {len(duplicate_skill_students)}")

In [ ]:
# --- 1c. Generate Jobs & Job Requirements ---
job_records = []
for i in range(NUM_JOBS):
    job_records.append({
        "job_id": f"job_{i}",
        "company_id": random.choice(companies),
        "seniority_level": random.randint(0, 5)  # 0=Intern, 5=Staff
    })
jobs_df = pd.DataFrame(job_records)
jobs_df.to_csv(f"{DATA_DIR}/jobs.csv", index=False)

job_skills_records = []
zero_req_jobs = set(random.sample(range(NUM_JOBS), 10))  # 10 jobs with zero requirements

for i in range(NUM_JOBS):
    if i in zero_req_jobs:
        continue
    num_reqs = random.randint(5, 15)
    req_skills = random.sample(skills_taxonomy, num_reqs)
    for skill in req_skills:
        min_prof = random.randint(30, 80)
        job_skills_records.append({"job_id": f"job_{i}", "skill_id": skill, "min_proficiency": min_prof})

job_skills_df = pd.DataFrame(job_skills_records)
job_skills_df.to_csv(f"{DATA_DIR}/job_skills.csv", index=False)

print(f"✅ Jobs: {len(jobs_df):,}")
print(f"✅ Job Skill Requirements: {len(job_skills_df):,}")
print(f"   └─ Zero-requirement jobs (edge case): {len(zero_req_jobs)}")

In [ ]:
# --- 1d. Simulate Historical Outcomes ---
# Pre-compute maps for fast lookup
student_exp_map = students_df.set_index("student_id")["years_of_experience"].to_dict()
job_sen_map = jobs_df.set_index("job_id")["seniority_level"].to_dict()

student_skills_grouped = student_skills_df.groupby("student_id").apply(
    lambda x: dict(zip(x["skill_id"], x["proficiency"]))
).to_dict()

job_skills_grouped = job_skills_df.groupby("job_id").apply(
    lambda x: dict(zip(x["skill_id"], x["min_proficiency"]))
).to_dict()

outcome_records = []
for i in range(NUM_STUDENTS):
    sid = f"student_{i}"
    s_exp = student_exp_map[sid]
    s_skills = student_skills_grouped.get(sid, {})
    applied_jobs = random.sample(range(NUM_JOBS), 15)

    for j in applied_jobs:
        jid = f"job_{j}"
        j_sen = job_sen_map[jid]
        j_skills = job_skills_grouped.get(jid, {})

        # Skill overlap
        if len(j_skills) == 0:
            overlap_ratio = 1.0
        else:
            matched = sum(1 for req_s, min_p in j_skills.items()
                         if req_s in s_skills and pd.notnull(s_skills[req_s]) and s_skills[req_s] >= min_p)
            overlap_ratio = matched / len(j_skills)

        # Experience fit
        exp_fit_score = 1.0 / (1.0 + abs(s_exp - j_sen))

        # Noisy ground truth
        noise = np.random.normal(0, 0.15)
        p_shortlist_raw = (0.6 * overlap_ratio) + (0.3 * exp_fit_score) + noise
        was_shortlisted = int(p_shortlist_raw > 0.5)
        p_hire_raw = p_shortlist_raw + np.random.normal(0, 0.1)
        was_hired = int(was_shortlisted and (p_hire_raw > 0.65))

        outcome_records.append({
            "student_id": sid, "job_id": jid,
            "was_shortlisted": was_shortlisted, "was_hired": was_hired
        })

outcomes_df = pd.DataFrame(outcome_records)
outcomes_df.to_csv(f"{DATA_DIR}/outcomes.csv", index=False)

print(f"\n✅ Outcomes (application records): {len(outcomes_df):,}")
print(f"   ├─ Shortlisted: {outcomes_df['was_shortlisted'].sum():,} ({outcomes_df['was_shortlisted'].mean():.1%})")
print(f"   └─ Hired:       {outcomes_df['was_hired'].sum():,} ({outcomes_df['was_hired'].mean():.1%})")
print("\n🏁 Data generation complete with edge cases injected.")

---
## 2. Exploratory Data Analysis (EDA)

Before building features, we examine the data distributions and quality.

In [ ]:
# Reload from CSV (simulates a fresh pipeline start)
students_df = pd.read_csv(f"{DATA_DIR}/students.csv")
student_skills_df = pd.read_csv(f"{DATA_DIR}/student_skills.csv")
jobs_df = pd.read_csv(f"{DATA_DIR}/jobs.csv")
job_skills_df = pd.read_csv(f"{DATA_DIR}/job_skills.csv")
outcomes_df = pd.read_csv(f"{DATA_DIR}/outcomes.csv")

print("📋 Dataset Shapes:")
for name, df in [("students", students_df), ("student_skills", student_skills_df),
                  ("jobs", jobs_df), ("job_skills", job_skills_df), ("outcomes", outcomes_df)]:
    print(f"   {name:>16}: {df.shape[0]:>8,} rows × {df.shape[1]} cols")

In [ ]:
# --- 2a. Target Variable Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Shortlisted distribution
shortlist_counts = outcomes_df['was_shortlisted'].value_counts()
colors_sl = ['#FF6B6B', '#4ECDC4']
axes[0].pie(shortlist_counts, labels=['Not Shortlisted', 'Shortlisted'],
            autopct='%1.1f%%', colors=colors_sl, startangle=90,
            explode=(0.05, 0), shadow=True, textprops={'fontsize': 12})
axes[0].set_title('Shortlisting Distribution', fontsize=14, fontweight='bold')

# Hired distribution
hire_counts = outcomes_df['was_hired'].value_counts()
colors_h = ['#FF6B6B', '#45B7D1']
axes[1].pie(hire_counts, labels=['Not Hired', 'Hired'],
            autopct='%1.1f%%', colors=colors_h, startangle=90,
            explode=(0.05, 0), shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Hiring Distribution (Target Variable)', fontsize=14, fontweight='bold')

plt.suptitle('Class Balance Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/class_balance.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"⚠️  Class imbalance: {outcomes_df['was_hired'].mean():.1%} positive rate — model must handle this.")

In [ ]:
# --- 2b. Student Distributions ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Experience distribution
sns.histplot(students_df['years_of_experience'], bins=10, kde=True, ax=axes[0], color='#6C5CE7')
axes[0].set_title('Years of Experience Distribution', fontweight='bold')
axes[0].set_xlabel('Years')

# Degree distribution
degree_counts = students_df['degree'].value_counts()
bars = axes[1].bar(degree_counts.index, degree_counts.values,
                    color=['#6C5CE7', '#00B894', '#FDCB6E', '#E17055'])
axes[1].set_title('Degree Distribution', fontweight='bold')
axes[1].set_xlabel('Degree')
axes[1].set_ylabel('Count')
for bar, val in zip(bars, degree_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontweight='bold')

# Skills per student
skills_per_student = student_skills_df.groupby('student_id').size()
sns.histplot(skills_per_student, bins=25, kde=True, ax=axes[2], color='#00B894')
axes[2].set_title('Skills per Student', fontweight='bold')
axes[2].set_xlabel('Number of Skills')
axes[2].axvline(skills_per_student.median(), color='red', linestyle='--', label=f'Median: {skills_per_student.median():.0f}')
axes[2].legend()

plt.suptitle('Student Profile Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/student_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 2c. Job & Seniority Analysis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Seniority level distribution
seniority_counts = jobs_df['seniority_level'].value_counts().sort_index()
seniority_labels = ['Intern (0)', 'Junior (1)', 'Mid (2)', 'Senior (3)', 'Lead (4)', 'Staff (5)']
colors_sen = plt.cm.viridis(np.linspace(0.2, 0.9, len(seniority_counts)))
axes[0].bar(seniority_labels[:len(seniority_counts)], seniority_counts.values, color=colors_sen)
axes[0].set_title('Jobs by Seniority Level', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Requirements per job
reqs_per_job = job_skills_df.groupby('job_id').size()
sns.histplot(reqs_per_job, bins=15, kde=True, ax=axes[1], color='#E17055')
axes[1].set_title('Skill Requirements per Job', fontweight='bold')
axes[1].set_xlabel('Number of Required Skills')
axes[1].axvline(reqs_per_job.median(), color='red', linestyle='--', label=f'Median: {reqs_per_job.median():.0f}')
axes[1].legend()

plt.suptitle('Job Landscape', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/job_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 2d. Data Quality Report ---
print("🔍 Data Quality Report")
print("=" * 50)

# Missing values
missing_prof = student_skills_df['proficiency'].isna().sum()
total_skills = len(student_skills_df)
print(f"\n📌 Missing proficiency scores: {missing_prof:,} / {total_skills:,} ({missing_prof/total_skills:.1%})")

# Duplicate skills
dupes = student_skills_df.groupby(['student_id', 'skill_id']).size()
num_dupes = (dupes > 1).sum()
print(f"📌 Duplicate (student, skill) entries: {num_dupes:,}")

# Cold-start students
all_student_ids = set(students_df['student_id'])
skilled_student_ids = set(student_skills_df['student_id'])
cold_start = all_student_ids - skilled_student_ids
print(f"📌 Cold-start students (zero skills): {len(cold_start)}")

# Zero-requirement jobs
all_job_ids = set(jobs_df['job_id'])
jobs_with_reqs = set(job_skills_df['job_id'])
zero_req = all_job_ids - jobs_with_reqs
print(f"📌 Zero-requirement jobs: {len(zero_req)}")

print("\n✅ All edge cases accounted for in feature engineering pipeline.")

---
## 3. Feature Engineering

We construct 4 ML-ready features from the raw data:

| Feature | Description | Range |
|---------|-------------|-------|
| `skill_overlap_ratio` | Fraction of job requirements met by the student | [0, 1] |
| `proficiency_gap` | Average gap between required and actual proficiency | [0, ∞) |
| `experience_fit` | How well student experience matches job seniority | (0, 1] |
| `college_hire_prior` | Historical hire rate of the student's college | [0, 1] |

**Key decisions:**
- College priors are computed on **training data only** to prevent data leakage
- Duplicate skills resolved by taking **max proficiency**
- Missing proficiency treated as **0** (worst case)

In [ ]:
class FeatureEngineer:
    """Feature engineering pipeline for the recommendation model."""

    def __init__(self):
        self.priors = {}
        self.student_map = {}
        self.job_map = {}
        self.student_skills_map = {}
        self.job_skills_map = {}

    def load_context(self, data_dir="data"):
        """Load context tables for fast feature lookup."""
        students = pd.read_csv(f"{data_dir}/students.csv")
        self.student_map = students.set_index("student_id").to_dict(orient="index")

        jobs = pd.read_csv(f"{data_dir}/jobs.csv")
        self.job_map = jobs.set_index("job_id").to_dict(orient="index")

        # Deduplicate student skills by taking max proficiency
        student_skills = pd.read_csv(f"{data_dir}/student_skills.csv")
        student_skills = student_skills.groupby(["student_id", "skill_id"])["proficiency"].max().reset_index()

        self.student_skills_map = student_skills.groupby("student_id").apply(
            lambda x: dict(zip(x["skill_id"], x["proficiency"]))
        ).to_dict()

        job_skills = pd.read_csv(f"{data_dir}/job_skills.csv")
        self.job_skills_map = job_skills.groupby("job_id").apply(
            lambda x: dict(zip(x["skill_id"], x["min_proficiency"]))
        ).to_dict()

    def fit(self, train_outcomes_df):
        """Learn priors from training data only to avoid leakage."""
        merged = train_outcomes_df.copy()
        merged["college_id"] = merged["student_id"].map(
            lambda x: self.student_map.get(x, {}).get("college_id")
        )
        self.priors["college_hire_rate"] = merged.groupby("college_id")["was_hired"].mean().to_dict()
        self.priors["global_hire_rate"] = merged["was_hired"].mean()

    def transform(self, pairs_df):
        """Transform (student_id, job_id) pairs into ML features."""
        features = []
        for _, row in pairs_df.iterrows():
            sid, jid = row["student_id"], row["job_id"]

            s_meta = self.student_map.get(sid, {"years_of_experience": 0, "college_id": "unknown"})
            j_meta = self.job_map.get(jid, {"seniority_level": 0})
            s_skills = self.student_skills_map.get(sid, {})
            j_skills = self.job_skills_map.get(jid, {})

            # Skill Overlap & Proficiency Gap
            if len(j_skills) == 0:
                overlap_ratio, prof_gap = 1.0, 0.0
            else:
                matched, gap_sum = 0, 0.0
                for req_skill, min_prof in j_skills.items():
                    if req_skill in s_skills:
                        actual = s_skills[req_skill]
                        if pd.isnull(actual):
                            actual = 0
                        if actual >= min_prof:
                            matched += 1
                        else:
                            gap_sum += (min_prof - actual)
                    else:
                        gap_sum += min_prof
                overlap_ratio = matched / len(j_skills)
                prof_gap = gap_sum / len(j_skills)

            # Experience Fit
            exp_fit = 1.0 / (1.0 + abs(s_meta["years_of_experience"] - j_meta["seniority_level"]))

            # College Prior
            college_id = s_meta["college_id"]
            college_prior = self.priors.get("college_hire_rate", {}).get(
                college_id, self.priors.get("global_hire_rate", 0.0)
            )

            features.append({
                "skill_overlap_ratio": overlap_ratio,
                "proficiency_gap": prof_gap,
                "experience_fit": exp_fit,
                "college_hire_prior": college_prior,
                "college_id": college_id,
                "seniority_level": j_meta["seniority_level"]
            })

        return pd.DataFrame(features)


# Initialize and load context
fe = FeatureEngineer()
fe.load_context(DATA_DIR)
print("✅ Feature Engineer loaded.")
print(f"   ├─ Students in context: {len(fe.student_map):,}")
print(f"   ├─ Jobs in context: {len(fe.job_map):,}")
print(f"   ├─ Students with skills: {len(fe.student_skills_map):,}")
print(f"   └─ Jobs with requirements: {len(fe.job_skills_map):,}")

---
## 4. Data Splitting (Group-Aware)

We use `GroupShuffleSplit` grouped by `student_id` to prevent a student's applications from leaking across train/val/test sets.

**Split ratio:** Train 70% | Validation 15% | Test 15%

In [ ]:
outcomes = pd.read_csv(f"{DATA_DIR}/outcomes.csv")

# Split 1: Train (70%) vs Temp (30%)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(outcomes, groups=outcomes["student_id"]))

train_outcomes = outcomes.iloc[train_idx]
temp_outcomes = outcomes.iloc[temp_idx]

# Split 2: Val (50% of 30% = 15%) vs Test (50% of 30% = 15%)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp_outcomes, groups=temp_outcomes["student_id"]))

val_outcomes = temp_outcomes.iloc[val_idx]
test_outcomes = temp_outcomes.iloc[test_idx]

# Save test set for later evaluation
test_outcomes.to_csv(f"{DATA_DIR}/test_outcomes.csv", index=False)

# Verify no student leakage
train_students = set(train_outcomes['student_id'])
val_students = set(val_outcomes['student_id'])
test_students = set(test_outcomes['student_id'])

assert len(train_students & val_students) == 0, "Leakage: train ∩ val"
assert len(train_students & test_students) == 0, "Leakage: train ∩ test"
assert len(val_students & test_students) == 0, "Leakage: val ∩ test"

print("📊 Data Split Summary:")
print(f"   ├─ Train: {len(train_outcomes):>6,} rows ({len(train_outcomes)/len(outcomes):.1%}) | {len(train_students):,} unique students")
print(f"   ├─ Val:   {len(val_outcomes):>6,} rows ({len(val_outcomes)/len(outcomes):.1%}) | {len(val_students):,} unique students")
print(f"   └─ Test:  {len(test_outcomes):>6,} rows ({len(test_outcomes)/len(outcomes):.1%}) | {len(test_students):,} unique students")
print("\n✅ No student leakage across splits confirmed.")

In [ ]:
# --- Fit on training data only, then transform all splits ---
fe.fit(train_outcomes)

print("Extracting features...")
X_train = fe.transform(train_outcomes)
y_train = train_outcomes["was_hired"].values

X_val = fe.transform(val_outcomes)
y_val = val_outcomes["was_hired"].values

X_test_full = fe.transform(test_outcomes)
y_test = test_outcomes["was_hired"].values

FEATURES = ["skill_overlap_ratio", "proficiency_gap", "experience_fit", "college_hire_prior"]

X_train_sub = X_train[FEATURES]
X_val_sub = X_val[FEATURES]
X_test_sub = X_test_full[FEATURES]

print(f"\n✅ Feature matrices ready:")
print(f"   ├─ X_train: {X_train_sub.shape}")
print(f"   ├─ X_val:   {X_val_sub.shape}")
print(f"   └─ X_test:  {X_test_sub.shape}")

In [ ]:
# --- Feature Distributions (Train set) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_feat = ['#6C5CE7', '#00B894', '#FDCB6E', '#E17055']

for i, (feat, color) in enumerate(zip(FEATURES, colors_feat)):
    ax = axes[i // 2][i % 2]
    # Separate by target class
    pos_vals = X_train_sub.loc[y_train == 1, feat]
    neg_vals = X_train_sub.loc[y_train == 0, feat]

    ax.hist(neg_vals, bins=40, alpha=0.6, label='Not Hired', color='#FF6B6B', density=True)
    ax.hist(pos_vals, bins=40, alpha=0.6, label='Hired', color='#4ECDC4', density=True)
    ax.set_title(feat, fontweight='bold', fontsize=13)
    ax.legend()
    ax.set_ylabel('Density')

plt.suptitle('Feature Distributions by Target Class (Train Set)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print("📈 Features show clear separation between classes — good discriminative power.")

In [ ]:
# --- Feature Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(8, 6))

corr_df = X_train_sub.copy()
corr_df['was_hired'] = y_train
corr_matrix = corr_df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, mask=mask,
            square=True, linewidths=1, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/correlation_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print("💡 Low inter-feature correlation = each feature contributes unique information.")

---
## 5. Baseline Model

A simple rule-based ranker that scores candidates **solely on skill overlap ratio**. This sets the performance floor that our ML model must beat.

In [ ]:
class BaselineRanker:
    """Rule-based ranker using only skill_overlap_ratio."""

    def predict_proba(self, features_df):
        p_true = features_df["skill_overlap_ratio"].values
        np.random.seed(42)
        p_true = np.clip(p_true + np.random.normal(0, 0.01, size=len(p_true)), 0, 1)
        p_false = 1.0 - p_true
        return np.vstack((p_false, p_true)).T


baseline = BaselineRanker()
y_base_val_proba = baseline.predict_proba(X_val)[:, 1]
y_base_val_pred = (y_base_val_proba > 0.5).astype(int)

base_auc = roc_auc_score(y_val, y_base_val_proba)
base_prec = precision_score(y_val, y_base_val_pred, zero_division=0)
base_rec = recall_score(y_val, y_base_val_pred, zero_division=0)

print(f"📊 Baseline Performance (Validation Set):")
print(f"   ├─ AUC-ROC:   {base_auc:.4f}")
print(f"   ├─ Precision:  {base_prec:.4f}")
print(f"   └─ Recall:     {base_rec:.4f}")

---
## 6. Model Training & Hyperparameter Sweep

We train a **GradientBoostingClassifier** with 5 hyperparameter configurations and select the best based on validation AUC-ROC.

In [ ]:
# Hyperparameter configurations to sweep
configs = [
    {"n_estimators": 50,  "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 100, "max_depth": 5, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 50,  "max_depth": 5, "learning_rate": 0.2},
]

best_auc = 0
best_model = None
best_config = None
experiment_log = []

print("🔄 Running Hyperparameter Sweep...")
print(f"{'Run':>4} | {'n_est':>5} | {'depth':>5} | {'lr':>6} | {'Val AUC':>8} | {'Val Prec':>8}")
print("-" * 56)

for i, config in enumerate(configs):
    model = GradientBoostingClassifier(**config, random_state=42)
    model.fit(X_train_sub, y_train)

    y_val_proba = model.predict_proba(X_val_sub)[:, 1]
    y_val_pred = (y_val_proba > 0.5).astype(int)

    auc = roc_auc_score(y_val, y_val_proba)
    prec = precision_score(y_val, y_val_pred, zero_division=0)

    log_entry = {"run": i, "params": config, "val_auc": auc, "val_precision": prec}
    experiment_log.append(log_entry)

    marker = " ⭐" if auc > best_auc else ""
    print(f"  {i:>2} | {config['n_estimators']:>5} | {config['max_depth']:>5} | {config['learning_rate']:>6.2f} | {auc:>8.4f} | {prec:>8.4f}{marker}")

    if auc > best_auc:
        best_auc = auc
        best_model = model
        best_config = config

# Save experiment log
with open("experiments.jsonl", "w") as f:
    for entry in experiment_log:
        f.write(json.dumps(entry) + "\n")

print(f"\n🏆 Best Model: AUC={best_auc:.4f} with {best_config}")

In [ ]:
# --- Visualize Hyperparameter Sweep Results ---
exp_df = pd.DataFrame(experiment_log)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC by run
colors_run = ['#4ECDC4' if row['val_auc'] == best_auc else '#95a5a6' for _, row in exp_df.iterrows()]
bars1 = axes[0].bar(exp_df['run'], exp_df['val_auc'], color=colors_run, edgecolor='white', linewidth=1.5)
axes[0].set_xlabel('Run')
axes[0].set_ylabel('Validation AUC-ROC')
axes[0].set_title('AUC-ROC by Configuration', fontweight='bold')
axes[0].set_ylim(exp_df['val_auc'].min() - 0.005, exp_df['val_auc'].max() + 0.005)
for bar, auc_val in zip(bars1, exp_df['val_auc']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                 f'{auc_val:.4f}', ha='center', fontsize=9, fontweight='bold')

# Precision by run
colors_run2 = ['#6C5CE7' if row['val_auc'] == best_auc else '#95a5a6' for _, row in exp_df.iterrows()]
bars2 = axes[1].bar(exp_df['run'], exp_df['val_precision'], color=colors_run2, edgecolor='white', linewidth=1.5)
axes[1].set_xlabel('Run')
axes[1].set_ylabel('Validation Precision')
axes[1].set_title('Precision by Configuration', fontweight='bold')
axes[1].set_ylim(exp_df['val_precision'].min() - 0.02, exp_df['val_precision'].max() + 0.02)
for bar, prec_val in zip(bars2, exp_df['val_precision']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{prec_val:.4f}', ha='center', fontsize=9, fontweight='bold')

# Add config labels
for ax in axes:
    ax.set_xticks(exp_df['run'])
    labels = [f"R{r}\n({c['n_estimators']},{c['max_depth']},{c['learning_rate']})" for r, c in zip(exp_df['run'], configs)]
    ax.set_xticklabels(labels, fontsize=8)

plt.suptitle('Hyperparameter Sweep Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/hyperparam_sweep.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 7. Model Evaluation (Held-Out Test Set)

Final evaluation on the test set comparing the ML model against the baseline.

In [ ]:
# --- Compute metrics helper ---
def compute_metrics(y_true, y_pred, y_proba):
    auc = roc_auc_score(y_true, y_proba)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return {"AUC": auc, "Precision": prec, "Recall": rec, "FPR": fpr, "TP": tp, "FP": fp, "TN": tn, "FN": fn}


# Predictions
y_base_test_proba = baseline.predict_proba(X_test_full)[:, 1]
y_base_test_pred = (y_base_test_proba > 0.5).astype(int)

y_mod_test_proba = best_model.predict_proba(X_test_sub)[:, 1]
y_mod_test_pred = (y_mod_test_proba > 0.5).astype(int)

metrics_base = compute_metrics(y_test, y_base_test_pred, y_base_test_proba)
metrics_mod = compute_metrics(y_test, y_mod_test_pred, y_mod_test_proba)

# Results table
print("\n" + "=" * 60)
print(" 📊 TEST SET EVALUATION — ML Model vs Baseline")
print("=" * 60)
print(f"{'Metric':<12} | {'Baseline':>10} | {'ML Model':>10} | {'Delta':>10}")
print("-" * 52)
for k in ["AUC", "Precision", "Recall", "FPR"]:
    delta = metrics_mod[k] - metrics_base[k]
    arrow = "↑" if delta > 0 and k != "FPR" else ("↓" if delta < 0 and k == "FPR" else ("↓" if delta < 0 else "↑"))
    good = (k != "FPR" and delta > 0) or (k == "FPR" and delta < 0)
    emoji = "✅" if good else "⚠️"
    print(f"{k:<12} | {metrics_base[k]:>10.4f} | {metrics_mod[k]:>10.4f} | {delta:>+10.4f} {emoji}")

In [ ]:
# --- ROC Curve Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve
fpr_base, tpr_base, _ = roc_curve(y_test, y_base_test_proba)
fpr_mod, tpr_mod, _ = roc_curve(y_test, y_mod_test_proba)

axes[0].plot(fpr_base, tpr_base, color='#FF6B6B', linewidth=2.5,
             label=f'Baseline (AUC = {metrics_base["AUC"]:.4f})')
axes[0].plot(fpr_mod, tpr_mod, color='#4ECDC4', linewidth=2.5,
             label=f'ML Model (AUC = {metrics_mod["AUC"]:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0].fill_between(fpr_mod, tpr_mod, alpha=0.1, color='#4ECDC4')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold', fontsize=14)
axes[0].legend(loc='lower right', framealpha=0.9)

# Precision-Recall Curve
prec_base, rec_base, _ = precision_recall_curve(y_test, y_base_test_proba)
prec_mod, rec_mod, _ = precision_recall_curve(y_test, y_mod_test_proba)

axes[1].plot(rec_base, prec_base, color='#FF6B6B', linewidth=2.5, label='Baseline')
axes[1].plot(rec_mod, prec_mod, color='#4ECDC4', linewidth=2.5, label='ML Model')
axes[1].fill_between(rec_mod, prec_mod, alpha=0.1, color='#4ECDC4')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold', fontsize=14)
axes[1].legend(loc='upper right', framealpha=0.9)

plt.suptitle('Model vs Baseline — Test Set Performance', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/roc_pr_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- Confusion Matrices Side by Side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_base = confusion_matrix(y_test, y_base_test_pred)
cm_mod = confusion_matrix(y_test, y_mod_test_pred)

for ax, cm, title, cmap in [
    (axes[0], cm_base, 'Baseline (Skill Overlap Only)', 'Reds'),
    (axes[1], cm_mod, f'ML Model ({best_config})', 'Greens')
]:
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Not Hired', 'Hired'],
                yticklabels=['Not Hired', 'Hired'],
                linewidths=2, linecolor='white',
                annot_kws={'size': 16, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(title, fontweight='bold', fontsize=12)

plt.suptitle('Confusion Matrix Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- Segment Analysis: By College ---
test_enriched = test_outcomes.copy()
test_enriched["college_id"] = X_test_full["college_id"].values
test_enriched["seniority"] = X_test_full["seniority_level"].values
test_enriched["base_proba"] = y_base_test_proba
test_enriched["mod_proba"] = y_mod_test_proba

print("📊 Segment Analysis — AUC by College (Top 5 by volume):")
print(f"{'College':<12} | {'Volume':>6} | {'Base AUC':>8} | {'ML AUC':>8} | {'Delta':>8}")
print("-" * 52)

college_aucs = []
top_colleges = test_enriched["college_id"].value_counts().head(5).index
for col in top_colleges:
    sub = test_enriched[test_enriched["college_id"] == col]
    try:
        b_auc = roc_auc_score(sub["was_hired"], sub["base_proba"])
        m_auc = roc_auc_score(sub["was_hired"], sub["mod_proba"])
        delta = m_auc - b_auc
        print(f"{col:<12} | {len(sub):>6} | {b_auc:>8.3f} | {m_auc:>8.3f} | {delta:>+8.3f}")
        college_aucs.append({"college": col, "volume": len(sub), "base_auc": b_auc, "ml_auc": m_auc})
    except ValueError:
        pass

print("\n📊 Segment Analysis — AUC by Seniority Level:")
print(f"{'Seniority':<12} | {'Volume':>6} | {'Base AUC':>8} | {'ML AUC':>8} | {'Delta':>8}")
print("-" * 52)

seniority_aucs = []
for sen in sorted(test_enriched["seniority"].unique()):
    sub = test_enriched[test_enriched["seniority"] == sen]
    try:
        b_auc = roc_auc_score(sub["was_hired"], sub["base_proba"])
        m_auc = roc_auc_score(sub["was_hired"], sub["mod_proba"])
        delta = m_auc - b_auc
        print(f"Level {sen:<6} | {len(sub):>6} | {b_auc:>8.3f} | {m_auc:>8.3f} | {delta:>+8.3f}")
        seniority_aucs.append({"seniority": sen, "volume": len(sub), "base_auc": b_auc, "ml_auc": m_auc})
    except ValueError:
        pass

In [ ]:
# --- Segment AUC Visualization ---
if seniority_aucs:
    sen_df = pd.DataFrame(seniority_aucs)

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(sen_df))
    width = 0.35

    bars1 = ax.bar(x - width/2, sen_df['base_auc'], width, label='Baseline', color='#FF6B6B', edgecolor='white')
    bars2 = ax.bar(x + width/2, sen_df['ml_auc'], width, label='ML Model', color='#4ECDC4', edgecolor='white')

    ax.set_xlabel('Seniority Level')
    ax.set_ylabel('AUC-ROC')
    ax.set_title('AUC-ROC by Seniority Level — Model vs Baseline', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    seniority_labels_map = {0: 'Intern', 1: 'Junior', 2: 'Mid', 3: 'Senior', 4: 'Lead', 5: 'Staff'}
    ax.set_xticklabels([f"{seniority_labels_map.get(int(s), s)} ({int(s)})" for s in sen_df['seniority']])
    ax.legend()
    ax.set_ylim(0.5, 1.0)

    # Annotate deltas
    for i, (b, m) in enumerate(zip(sen_df['base_auc'], sen_df['ml_auc'])):
        delta = m - b
        ax.annotate(f'{delta:+.3f}', xy=(i + width/2, m + 0.005),
                    ha='center', fontsize=9, fontweight='bold', color='#2d3436')

    plt.tight_layout()
    plt.savefig('outputs/segment_auc.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## 8. Explainability

Understanding **why** the model makes certain predictions is critical for a placement platform.

In [ ]:
# --- 8a. Feature Importance (from GradientBoosting) ---
importances = best_model.feature_importances_
feat_importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors_imp = ['#6C5CE7', '#00B894', '#FDCB6E', '#E17055']
bars = ax.barh(feat_importance['Feature'], feat_importance['Importance'],
               color=colors_imp, edgecolor='white', linewidth=1.5, height=0.6)

for bar, val in zip(bars, feat_importance['Importance']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('GradientBoosting Feature Importances', fontweight='bold', fontsize=14)
ax.set_xlim(0, max(importances) * 1.2)
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print("\n📊 Feature Importance Ranking:")
for _, row in feat_importance.sort_values('Importance', ascending=False).iterrows():
    print(f"   {row['Feature']:<24} {row['Importance']:.4f}")

In [ ]:
# --- 8b. Plain-English Explanations ---
def generate_explanation(features_row):
    """Generate a human-readable explanation for a prediction."""
    reasons = []

    overlap = features_row.get("skill_overlap_ratio", 0)
    if overlap == 1.0:
        reasons.append("You possess all the required skills for this role.")
    elif overlap >= 0.7:
        reasons.append(f"You have a strong skill match ({overlap:.0%} of required skills).")
    elif overlap > 0.0:
        reasons.append(f"You have a partial skill match ({overlap:.0%} of required skills).")
    else:
        reasons.append("You do not have any of the required skills verified.")

    gap = features_row.get("proficiency_gap", 0)
    if gap <= 0 and overlap > 0:
        reasons.append("Your proficiency exceeds the minimum requirements.")
    elif gap > 0:
        reasons.append("There is a gap in your proficiency levels compared to what is expected.")

    exp = features_row.get("experience_fit", 0)
    if exp == 1.0:
        reasons.append("Your years of experience perfectly align with the seniority of this role.")
    elif exp >= 0.5:
        reasons.append("Your experience level is close to the expected seniority.")
    else:
        reasons.append("Your experience level differs significantly from the expected seniority.")

    return " ".join(reasons)


# Demo: explain top 5 predictions from the test set
X_test_full_copy = X_test_full.copy()
X_test_full_copy['score'] = y_mod_test_proba
X_test_full_copy['student_id'] = test_outcomes['student_id'].values
X_test_full_copy['job_id'] = test_outcomes['job_id'].values
X_test_full_copy['actual'] = y_test

top_5 = X_test_full_copy.sort_values('score', ascending=False).head(5)

print("🔍 Top 5 Predictions with Explanations:\n")
for i, (_, row) in enumerate(top_5.iterrows(), 1):
    print(f"  {i}. {row['student_id']} → {row['job_id']}")
    print(f"     Score: {row['score']:.4f} | Actual: {'Hired ✅' if row['actual'] == 1 else 'Not Hired ❌'}")
    print(f"     💡 {generate_explanation(row.to_dict())}")
    print()

---
## 9. Model Persistence

Save the best model artifact for deployment in the API serving layer.

In [ ]:
# Save model artifact
artifact = {
    "model": best_model,
    "features_to_use": FEATURES,
    "college_priors": fe.priors  # Must persist the priors computed during training
}

joblib.dump(artifact, "model.pkl")

# Verify it loads correctly
loaded = joblib.load("model.pkl")
assert loaded["features_to_use"] == FEATURES
assert loaded["model"] is not None
assert "college_hire_rate" in loaded["college_priors"]

model_size = os.path.getsize("model.pkl") / 1024
print(f"✅ Model saved to model.pkl ({model_size:.1f} KB)")
print(f"   ├─ Model type: {type(best_model).__name__}")
print(f"   ├─ Best params: {best_config}")
print(f"   ├─ Features: {FEATURES}")
print(f"   └─ College priors: {len(fe.priors.get('college_hire_rate', {}))} colleges")

---
## 10. Pipeline Summary & Final Report

In [ ]:
# --- Final Summary Dashboard ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (1) Metrics comparison bar chart
metrics_names = ['AUC', 'Precision', 'Recall']
base_vals = [metrics_base[m] for m in metrics_names]
mod_vals = [metrics_mod[m] for m in metrics_names]

x = np.arange(len(metrics_names))
axes[0, 0].bar(x - 0.2, base_vals, 0.35, label='Baseline', color='#FF6B6B', edgecolor='white')
axes[0, 0].bar(x + 0.2, mod_vals, 0.35, label='ML Model', color='#4ECDC4', edgecolor='white')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics_names)
axes[0, 0].set_title('Key Metrics Comparison', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 1.1)

# (2) Score distribution
axes[0, 1].hist(y_mod_test_proba[y_test == 0], bins=50, alpha=0.6, label='Not Hired', color='#FF6B6B', density=True)
axes[0, 1].hist(y_mod_test_proba[y_test == 1], bins=50, alpha=0.6, label='Hired', color='#4ECDC4', density=True)
axes[0, 1].axvline(0.5, color='black', linestyle='--', alpha=0.5, label='Threshold (0.5)')
axes[0, 1].set_title('Model Score Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Predicted Probability')
axes[0, 1].legend()

# (3) Feature importance
feat_imp_sorted = feat_importance.sort_values('Importance', ascending=True)
axes[1, 0].barh(feat_imp_sorted['Feature'], feat_imp_sorted['Importance'],
                color=['#6C5CE7', '#00B894', '#FDCB6E', '#E17055'], edgecolor='white', height=0.5)
axes[1, 0].set_title('Feature Importances', fontweight='bold')

# (4) ROC
axes[1, 1].plot(fpr_mod, tpr_mod, color='#4ECDC4', linewidth=2.5,
                label=f'ML Model (AUC={metrics_mod["AUC"]:.4f})')
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1, 1].fill_between(fpr_mod, tpr_mod, alpha=0.15, color='#4ECDC4')
axes[1, 1].set_xlabel('FPR')
axes[1, 1].set_ylabel('TPR')
axes[1, 1].set_title('ROC Curve', fontweight='bold')
axes[1, 1].legend()

plt.suptitle('🎯 Placemux ML Pipeline — Final Report', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/final_report.png', bbox_inches='tight', dpi=150)
plt.show()

print("\n" + "=" * 60)
print(" 🏁 ML PIPELINE COMPLETE")
print("=" * 60)
print(f"\n  📁 Artifacts:")
print(f"     ├─ model.pkl          — Trained model + features + priors")
print(f"     ├─ experiments.jsonl   — Hyperparameter sweep log")
print(f"     ├─ data/              — Generated datasets")
print(f"     └─ outputs/           — Visualizations")
print(f"\n  🏆 Best Model: GradientBoosting {best_config}")
print(f"  📈 Test AUC:  {metrics_mod['AUC']:.4f} (vs Baseline {metrics_base['AUC']:.4f})")
print(f"  🎯 Test Prec: {metrics_mod['Precision']:.4f} (vs Baseline {metrics_base['Precision']:.4f})")
print(f"  📊 Test Rec:  {metrics_mod['Recall']:.4f} (vs Baseline {metrics_base['Recall']:.4f})")